# Phase 2 matched experiment: saved-results analysis

This notebook analyzes the existing `phase3a_stable_cohort` experiment, including its validated one-row-per-minute matched trace, communication accounting, saved root-state correctness, synchronization intervals, and canonical event fields. It reads only the current saved summary, matched trace, synchronization events, configuration, and validation evidence; it does not run monitoring, tests, replay, or any experiment, and it writes no files.

Sources: the locked Phase 3A experiment section in [project_config.yaml](../config/project_config.yaml), monitoring preparation embedded in the current saved summary, [shared Phase 2 monitoring engine](shared_monitoring_engine.py), [current saved summary and matched trace](../artifacts/phase2_outputs/), and the existing [Phase 3A comparator analysis](phase3_analyze.ipynb). The field semantics and limitations are documented in [README_scripts.md](README_scripts.md#phase-2-two-sided-monitoring-us-323) and the engine's event export.

Run all cells from a clean kernel with the project Python environment. Only the canonical Phase 2 summary and operational evidence are accepted. Missing, stale, malformed, or mismatched evidence fails before scientific results are displayed. There is no root/archive fallback.

The independent validation receipt must be generated by `phase2_test.ipynb` before this read-only notebook can display results.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "project_config.yaml").is_file():
    parent = ROOT.parent
    if parent == ROOT:
        raise FileNotFoundError("Run this notebook from the sensor_project repository.")
    ROOT = parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.shared_monitoring_engine import load_phase3a_config, load_validated
CURRENT_DIR = ROOT / "artifacts" / "phase2_outputs"
SUMMARY_PATH = CURRENT_DIR / "phase2_summary.json"
TRACE_PATH = CURRENT_DIR / "phase2_matched_trace.csv"
EVENTS_PATH = ROOT / "artifacts" / "phase2_operational" / "phase2_events.csv"
CONFIG_PATH = ROOT / "config" / "project_config.yaml"
HISTORICAL_REFERENCE = (
    ROOT / "artifacts" / "phase3_operational" / "references"
    / "phase3a_phase2_baseline.json"
)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print(f"Repository: {ROOT}")
print(f"Current Phase 2 results: {CURRENT_DIR}")

## 1. Validate current evidence before displaying results

The gates below bind the saved result to the locked experiment, authoritative full-precision cohort P90, aligned input, historical reference, and shared monitoring source. They also validate the matched-trace digest/schema and approved UTC grid, the event-file digest/schema and per-event accounting, and the saved root-correctness status. No tables of scientific results are displayed until all gates pass.

The shared loader checks the independent receipt against the exact saved summary, matched trace, events, sources, tests, and inputs. This notebook does not rerun tests.

In [ ]:
if not CURRENT_DIR.is_dir():
    raise FileNotFoundError(f"Current Phase 2 results directory is missing: {CURRENT_DIR}")
for required_path in (SUMMARY_PATH, TRACE_PATH, EVENTS_PATH, CONFIG_PATH, HISTORICAL_REFERENCE):
    if not required_path.is_file():
        raise FileNotFoundError(f"Missing required current Phase 2 evidence: {required_path}")

summary = load_validated(ROOT)
config = load_phase3a_config(ROOT)
required_summary_fields = {
    "experiment_name", "input_source", "selected_sensor_ids", "start_timestamp_utc",
    "end_timestamp_utc", "endpoint_convention", "boundary_convention", "story",
    "algorithm", "threshold_policy", "threshold_source_series",
    "monitoring_preparation", "threshold_statistic", "threshold_value",
    "bucket_count", "global_threshold_violation_count", "false_safe_count",
    "false_violated_count", "root_state_not_evaluable_count",
    "root_state_evaluation_count", "correctness_status", "message_category_totals",
    "total_messages", "message_counts_reconcile", "resynchronization_count",
    "initialization_count", "monitoring_triggered_synchronization_count",
    "local_trigger_transmission_count", "membership_change_event_count",
    "membership_triggered_synchronization_count",
    "centralized_sensor_transmission_reference", "events_artifact", "events_sha256",
    "trace_artifact", "trace_sha256", "trace_row_count", "trace_columns",
    "provenance", "limitations",
}
missing_fields = required_summary_fields - set(summary)
if missing_fields:
    raise ValueError(f"Current Phase 2 summary is missing fields: {sorted(missing_fields)}")

if config["experiment_name"] != "phase3a_stable_cohort":
    raise ValueError("The experiment configuration is not the approved matched cohort.")
for key in (
    "experiment_name", "input_source", "selected_sensor_ids", "start_timestamp_utc",
    "end_timestamp_utc", "endpoint_convention", "boundary_convention",
):
    if summary[key] != config[key]:
        raise ValueError(f"Saved Phase 2 experiment does not match config field {key}.")
if summary["algorithm"] != "phase2" or summary["story"] != "US-3.23":
    raise ValueError("Saved result does not identify the expected Phase 2 producer schema.")
if summary["threshold_policy"] != "two_sided_absolute_distance_equal_allowances":
    raise ValueError("Saved result does not use the documented matched two-sided policy.")
threshold_config = config["threshold"]
if (
    summary["threshold_source_series"] != threshold_config["source_series"]
    or summary["threshold_statistic"] != "p90"
):
    raise ValueError("Saved threshold provenance differs from the locked configuration.")

input_path = ROOT / config["input_source"]
preparation = summary["monitoring_preparation"]
threshold = preparation["p90_value"]
if not math.isfinite(threshold):
    raise ValueError("Saved monitoring P90 must be finite.")
if summary["threshold_value"] != threshold:
    raise ValueError("Saved Phase 2 result uses a stale or different threshold.")

provenance = summary["provenance"]
expected_provenance = {
    "input_sha256": input_path,
    "config_sha256": CONFIG_PATH,
    "code_sha256": ROOT / "scripts" / "shared_monitoring_engine.py",
    "historical_baseline_sha256": HISTORICAL_REFERENCE,
}
for field, path in expected_provenance.items():
    if field not in provenance or sha256(path) != provenance[field]:
        raise ValueError(f"Saved Phase 2 provenance is stale for {path.relative_to(ROOT)}.")
if summary["events_artifact"] != "../phase2_operational/phase2_events.csv":
    raise ValueError("Saved event artifact reference does not name the current event CSV.")
if sha256(EVENTS_PATH) != summary["events_sha256"]:
    raise ValueError("Current event CSV does not match its saved SHA-256.")
event_columns = [
    "bucket_epoch", "bucket_time_utc", "event_resync_causes",
    "event_triggering_sensor_names", "entry_xbar_t0", "exit_xbar_t0_if_resync",
    "entry_root_allowance", "exit_root_allowance_if_resync", "root_known_state",
    "oracle_global_state", "trigger_message_count", "request_message_count",
    "response_message_count", "broadcast_message_count", "total_message_count",
]
events = pd.read_csv(EVENTS_PATH)
trace = pd.read_csv(TRACE_PATH)
if trace.columns.tolist() != summary["trace_columns"]:
    raise ValueError("Current matched trace schema or column order differs from the saved summary.")
if summary["trace_artifact"] != "phase2_matched_trace.csv":
    raise ValueError("Saved trace artifact reference does not name the current matched trace.")
expected_grid = pd.date_range(
    start=pd.to_datetime(config["start_timestamp_utc"], utc=True),
    end=pd.to_datetime(config["end_timestamp_utc"], utc=True),
    freq="min",
)
expected_epochs = expected_grid.map(lambda value: int(value.timestamp()))
trace_times = pd.to_datetime(trace["bucket_time_utc"], utc=True, errors="raise")
trace_epochs = trace["bucket_epoch"].astype("int64")
if (
    len(trace) != config["expected_bucket_count"]
    or len(expected_grid) != config["expected_bucket_count"]
    or len(trace) != summary["trace_row_count"]
    or len(trace) != summary["bucket_count"]
):
    raise ValueError("Matched trace row count differs from the approved evaluation window.")
if trace_epochs.duplicated().any() or not trace_epochs.is_monotonic_increasing:
    raise ValueError("Matched trace buckets are duplicated or out of order.")
if not trace_epochs.eq(expected_epochs.to_numpy()).all() or not pd.DatetimeIndex(trace_times).equals(expected_grid):
    raise ValueError("Matched trace timestamps do not match the approved one-minute UTC grid.")
if not trace_epochs.eq(trace_times.map(lambda value: int(value.timestamp()))).all():
    raise ValueError("Matched trace epochs do not match their UTC timestamps.")
trace_message_columns = list(summary["message_category_totals"])
if not trace[trace_message_columns].sum(axis=1).eq(trace["total_message_count"]).all():
    raise ValueError("One or more matched-trace message totals do not reconcile.")
if trace[trace_message_columns].sum().astype(int).to_dict() != summary["message_category_totals"]:
    raise ValueError("Matched-trace message categories differ from the saved summary.")
if int(trace["total_message_count"].sum()) != summary["total_messages"]:
    raise ValueError("Matched-trace transmissions differ from the saved total.")
if events.columns.tolist() != event_columns:
    raise ValueError("Current synchronization event CSV schema or column order is invalid.")
if len(events) != summary["resynchronization_count"]:
    raise ValueError("Synchronization event rows do not match the saved synchronization count.")
if events["bucket_epoch"].duplicated().any() or not events["bucket_epoch"].is_monotonic_increasing:
    raise ValueError("Synchronization event timestamps are duplicated or out of order.")
event_times = pd.to_datetime(events["bucket_time_utc"], utc=True, errors="raise")
event_epochs = event_times.map(lambda value: int(value.timestamp()))
if not events["bucket_epoch"].astype("int64").eq(event_epochs).all():
    raise ValueError("Synchronization event epochs do not match their UTC timestamps.")
if not events["bucket_epoch"].isin(trace_epochs).all():
    raise ValueError("A synchronization timestamp falls outside the matched evaluation grid.")
if int(events["bucket_epoch"].iloc[0]) != int(trace_epochs.iloc[0]):
    raise ValueError("Initialization is not the first synchronization at the window start.")
if not events["root_known_state"].isin(["SAFE", "VIOLATED"]).all():
    raise ValueError("A completed synchronization has no valid root-known state.")
if not events["oracle_global_state"].isin(["SAFE", "VIOLATED"]).all():
    raise ValueError("A completed synchronization has no valid oracle state.")
if not events["root_known_state"].eq(events["oracle_global_state"]).all():
    raise ValueError("Saved synchronization root states disagree with the offline oracle.")
message_columns = [
    "trigger_message_count", "request_message_count",
    "response_message_count", "broadcast_message_count",
]
if not events[message_columns].sum(axis=1).eq(events["total_message_count"]).all():
    raise ValueError("One or more synchronization event transmission totals do not reconcile.")
if events[message_columns].sum().astype(int).to_dict() != summary["message_category_totals"]:
    raise ValueError("Event transmission categories differ from the saved summary.")
if int(events["total_message_count"].sum()) != summary["total_messages"]:
    raise ValueError("Event transmissions differ from the saved total.")

expected_buckets = config["expected_bucket_count"]
if summary["bucket_count"] != expected_buckets:
    raise ValueError("Saved Phase 2 bucket count differs from the locked experiment.")
if summary["correctness_status"] != "passed":
    raise ValueError("Saved independent root-state correctness status is not passed.")
if (
    summary["root_state_evaluation_count"] != expected_buckets
    or summary["false_safe_count"] != 0
    or summary["false_violated_count"] != 0
    or summary["root_state_not_evaluable_count"] != 0
):
    raise ValueError("Saved matched root-state correctness counts do not pass the gate.")
if summary["message_counts_reconcile"] is not True:
    raise ValueError("Saved Phase 2 summary does not report reconciled message counts.")
if summary["total_messages"] != sum(summary["message_category_totals"].values()):
    raise ValueError("Saved Phase 2 message categories do not sum to the total.")
if summary["centralized_sensor_transmission_reference"] != expected_buckets * len(config["selected_sensor_ids"]):
    raise ValueError("Saved centralized transmission reference differs from the matched cohort.")
if summary["initialization_count"] != 1 or summary["membership_change_event_count"] != 0:
    raise ValueError("Saved synchronization diagnostics differ from the locked stable cohort.")
if summary["trace_sha256"] != sha256(TRACE_PATH):
    raise ValueError("Current matched trace does not match the summary SHA-256.")

validation_status = pd.DataFrame([
    {"check": "Saved root-state correctness", "status": summary["correctness_status"]},
    {"check": "Matched inputs, code, references, and protected files", "status": "passed"},
    {"check": "Matched trace grid, digest, summary, and message accounting", "status": "passed"},
    {"check": "Event schema, on-grid timestamps, oracle states, and accounting", "status": "passed"},
    {"check": "Independent regression and reproduction receipt", "status": "passed"},
])
print(f"Validated current experiment: {summary['experiment_name']}")
display(validation_status)

## 2. Experiment and policy

The matched run covers the fixed A/C/D/E/F/G cohort over the inclusive UTC interval in the table below. The threshold is the configured cohort P90, read at round-trip precision. SAFE is below threshold; VIOLATED includes equality. The shared policy uses equal allowances based on the absolute distance between the synchronized mean and threshold; local triggers test absolute deviation against that allowance. Synchronization updates apply to the following bucket. These are the existing US-3.23 rules; this notebook does not recompute the monitor.

In [ ]:
experiment_table = pd.DataFrame([
    {"item": "Experiment", "saved value": summary["experiment_name"]},
    {"item": "Sensors", "saved value": ", ".join(summary["selected_sensor_ids"])},
    {"item": "Interval (UTC, inclusive)", "saved value": f"{summary['start_timestamp_utc']} through {summary['end_timestamp_utc']}"},
    {"item": "Sampling", "saved value": f"{summary['bucket_count']:,} consecutive one-minute buckets"},
    {"item": "Threshold policy", "saved value": summary["threshold_policy"]},
    {"item": "Threshold series / statistic", "saved value": f"{summary['threshold_source_series']} / {summary['threshold_statistic'].upper()}"},
    {"item": "Threshold (full precision)", "saved value": repr(summary["threshold_value"])},
    {"item": "Boundary", "saved value": "SAFE < threshold; VIOLATED >= threshold"},
])
display(experiment_table)

## 3. Communication totals and the existing centralized reference

The category totals below come from the validated matched trace and reconcile to the saved summary. The per-minute column divides each transmission count by all 19,984 evaluated buckets. The comparison uses the existing centralized reference of one sensor transmission per configured sensor per bucket (also used in the Phase 3A comparator table). The displayed percent is `(reference - Phase 2 transmissions) / reference * 100`; it compares transmission counts, not equivalent protocols or universal performance.

In [ ]:
message_labels = {
    "trigger_message_count": "Local-trigger transmissions",
    "request_message_count": "Sensor-to-root requests",
    "response_message_count": "Root-to-sensor responses",
    "broadcast_message_count": "Root-to-sensor state-update broadcasts",
}
communication_table = pd.DataFrame([
    {
        "measure": message_labels[name],
        "transmissions": count,
        "per evaluated minute": count / summary["bucket_count"],
    }
    for name, count in trace[trace_message_columns].sum().astype(int).items()
])
communication_table.loc[len(communication_table)] = {
    "measure": "Phase 2 total",
    "transmissions": summary["total_messages"],
    "per evaluated minute": summary["total_messages"] / summary["bucket_count"],
}
reference = summary["centralized_sensor_transmission_reference"]
phase2_total = summary["total_messages"]
communication_comparison = pd.DataFrame([
    {"reference": "Centralized: one transmission per sensor per bucket", "transmissions": reference},
    {"reference": "Matched Phase 2 actual transmissions", "transmissions": phase2_total},
    {"reference": "Difference from centralized reference", "transmissions": reference - phase2_total},
    {"reference": "Reduction vs centralized reference", "transmissions": f"{(reference - phase2_total) / reference * 100:.2f}%"},
])
display(communication_table)
display(communication_comparison)

## 4. Root-state correctness and synchronization diagnostics

Root-state correctness is evaluated against an offline oracle computed from the centralized matched observations; the oracle does not control communication. The saved `false_safe_count` and `false_violated_count` are root-state mismatches, not synchronization-based warning metrics. The event CSV contains completed synchronization rows only, so it is not a substitute for the per-bucket correctness denominator. The summary records the full evaluated bucket count and explicit correctness status.

No legacy synchronization-based false-safe/false-alert metrics are relabeled or presented here. The archived full-dataset workflow and the historical signed-margin benchmark are not current results and are not used as denominators.

In [ ]:
correctness_table = pd.DataFrame([
    {"measure": "Saved root-state correctness status", "value": summary["correctness_status"]},
    {"measure": "Buckets independently evaluated", "value": summary["root_state_evaluation_count"]},
    {"measure": "Centralized threshold-violated buckets", "value": summary["global_threshold_violation_count"]},
    {"measure": "Root false-safe mismatches", "value": summary["false_safe_count"]},
    {"measure": "Root false-violated mismatches", "value": summary["false_violated_count"]},
    {"measure": "Root state not evaluable", "value": summary["root_state_not_evaluable_count"]},
])
synchronization_table = pd.DataFrame([
    {"measure": "Completed root synchronizations", "value": summary["resynchronization_count"]},
    {"measure": "Initialization synchronizations", "value": summary["initialization_count"]},
    {"measure": "Synchronizations triggered by local monitoring", "value": summary["monitoring_triggered_synchronization_count"]},
    {"measure": "Membership-change events", "value": summary["membership_change_event_count"]},
    {"measure": "Membership-triggered synchronizations", "value": summary["membership_triggered_synchronization_count"]},
])
display(correctness_table)
display(synchronization_table)

## 5. Existing synchronization event evidence

The table below is the complete saved synchronization-event CSV, not a newly generated trace. It contains one row per completed synchronization, with entry/exit reference values, causes, trigger sensors, root/oracle state, and actual message categories. Quiet-minute analysis uses the full matched trace; this event-only table is not used to infer quiet minutes.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(events)

## 6. Event-field guide

Definitions follow the current event schema in `scripts/shared_monitoring_engine.py`. Blank reference/allowance fields at initialization mean there was no prior synchronized value.

In [ ]:
event_field_guide = pd.DataFrame([
    {"field": "bucket_epoch / bucket_time_utc", "meaning": "Unix-second epoch and ISO-8601 UTC timestamp of the completed synchronization bucket."},
    {"field": "event_resync_causes", "meaning": "Recorded reason(s) for this synchronization, such as initialization or local-constraint violation."},
    {"field": "event_triggering_sensor_names", "meaning": "Sensors whose local checks requested this batched root synchronization."},
    {"field": "entry_xbar_t0 / exit_xbar_t0_if_resync", "meaning": "Root reference mean on entry and the newly synchronized mean, respectively."},
    {"field": "entry_root_allowance / exit_root_allowance_if_resync", "meaning": "Absolute threshold distance at entry and after synchronization; the updated allowance applies on the next bucket."},
    {"field": "root_known_state", "meaning": "Monitor's synchronized root state for the event bucket."},
    {"field": "oracle_global_state", "meaning": "Offline centralized-observation truth for validation; never a monitor input or communication trigger."},
    {"field": "trigger_message_count", "meaning": "Messages emitted by locally triggering sensors at this event."},
    {"field": "request_message_count / response_message_count", "meaning": "Sensor-to-root requests and root-to-sensor responses for participating sensors."},
    {"field": "broadcast_message_count", "meaning": "Root-to-sensor state-update transmissions to participating sensors."},
    {"field": "total_message_count", "meaning": "Actual event transmissions; equals the four event-level message categories."},
])
display(event_field_guide)

## 7. Communication activity and synchronization intervals (US-4.02)

Communication-active and message-free minutes are counted once per bucket from the full validated matched trace, regardless of how many trigger notices or transmissions a bucket contains. Synchronization gaps use consecutive completed root-synchronization timestamps; initialization is the first event at the start of the evaluation window. The final unsynchronized tail is reported separately rather than added as an interval. These gaps describe synchronization cadence, not network latency.

In [ ]:
active_minutes = int(trace["total_message_count"].gt(0).sum())
quiet_minutes = int(trace["total_message_count"].eq(0).sum())
if active_minutes + quiet_minutes != summary["bucket_count"]:
    raise ValueError("Communication-active and message-free minutes do not cover the evaluation window.")
activity_table = pd.DataFrame([
    {"minute class": "Communication-active", "minutes": active_minutes, "share of evaluation": active_minutes / summary["bucket_count"] * 100},
    {"minute class": "Message-free", "minutes": quiet_minutes, "share of evaluation": quiet_minutes / summary["bucket_count"] * 100},
])
display(activity_table)

trigger_lists = events["event_triggering_sensor_names"].fillna("").map(
    lambda value: [sensor.strip() for sensor in str(value).split("|") if sensor.strip()]
)
trigger_sensor_counts = pd.Series([len(sensors) for sensors in trigger_lists], index=events.index)
if not trigger_sensor_counts.eq(events["trigger_message_count"]).all():
    raise ValueError("Triggering-sensor names do not reconcile with event trigger-message counts.")
if int(trigger_sensor_counts.sum()) != summary["local_trigger_transmission_count"]:
    raise ValueError("Triggering-sensor appearances do not reconcile with local-trigger transmissions.")
triggered_sync_count = int(trigger_sensor_counts.gt(0).sum())
co_triggered_sync_count = int(trigger_sensor_counts.gt(1).sum())
operation_table = pd.DataFrame([
    {"operation": "Completed root synchronizations, including initialization", "count": summary["resynchronization_count"]},
    {"operation": "Initialization synchronizations", "count": summary["initialization_count"]},
    {"operation": "Local-monitoring-triggered synchronizations", "count": summary["monitoring_triggered_synchronization_count"]},
    {"operation": "Membership-triggered synchronizations", "count": summary["membership_triggered_synchronization_count"]},
    {"operation": "Synchronizations with at least one triggering sensor", "count": triggered_sync_count},
    {"operation": "Synchronizations with co-triggering sensors", "count": co_triggered_sync_count},
])
display(operation_table)

sensor_order = summary["selected_sensor_ids"]
sensor_trigger_counts = pd.Series(
    [sensor for sensors in trigger_lists for sensor in sensors]
).value_counts().reindex(sensor_order, fill_value=0)
sensor_trigger_table = sensor_trigger_counts.rename_axis("sensor").reset_index(name="triggering events")
display(sensor_trigger_table)
trigger_size_table = (
    trigger_sensor_counts.value_counts().sort_index().rename_axis("triggering sensors at synchronization")
    .reset_index(name="synchronization events")
)
display(trigger_size_table)

sync_gap_minutes = events["bucket_epoch"].astype("int64").diff().dropna() / 60
if len(sync_gap_minutes) != len(events) - 1 or sync_gap_minutes.le(0).any():
    raise ValueError("Completed synchronization timestamps do not define valid positive intervals.")
trailing_minutes = (int(trace_epochs.iloc[-1]) - int(events["bucket_epoch"].iloc[-1])) / 60
if trailing_minutes < 0:
    raise ValueError("The final synchronization occurs after the final evaluated bucket.")
gap_table = pd.DataFrame([
    {"measure": "Consecutive-synchronization intervals (event rows - 1)", "minutes": len(sync_gap_minutes)},
    {"measure": "Median synchronization gap", "minutes": sync_gap_minutes.median()},
    {"measure": "P75 synchronization gap", "minutes": sync_gap_minutes.quantile(0.75)},
    {"measure": "P90 synchronization gap", "minutes": sync_gap_minutes.quantile(0.90)},
    {"measure": "P95 synchronization gap", "minutes": sync_gap_minutes.quantile(0.95)},
    {"measure": "Maximum synchronization gap", "minutes": sync_gap_minutes.max()},
    {"measure": "Trailing time: final synchronization to final evaluated bucket", "minutes": trailing_minutes},
])
display(gap_table)

fig, ax = plt.subplots(figsize=(7.2, 3.8))
histogram_bins = np.geomspace(1, max(2.0, float(sync_gap_minutes.max())), 20)
ax.hist(sync_gap_minutes, bins=histogram_bins, color="#4472C4", edgecolor="white")
median_gap = float(sync_gap_minutes.median())
p90_gap = float(sync_gap_minutes.quantile(0.90))
ax.axvline(median_gap, color="#C00000", linestyle="--", label=f"Median: {median_gap:g} min")
ax.axvline(p90_gap, color="#ED7D31", linestyle=":", label=f"P90: {p90_gap:g} min")
ax.set_xscale("log")
ax.set_xlabel("Minutes between completed root synchronizations (log scale)")
ax.set_ylabel("Number of intervals")
ax.set_title("P2.2. Root-synchronization interval distribution")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
evaluated_by_weekday = trace_times.dt.day_name().value_counts().reindex(weekday_order, fill_value=0)
syncs_by_weekday = event_times.dt.day_name().value_counts().reindex(weekday_order, fill_value=0)
weekday_table = pd.DataFrame({
    "UTC weekday": weekday_order,
    "evaluated minutes": evaluated_by_weekday.to_numpy(),
    "synchronizations": syncs_by_weekday.to_numpy(),
})
weekday_table["synchronizations per 1,000 evaluated minutes"] = (
    weekday_table["synchronizations"] / weekday_table["evaluated minutes"] * 1000
)
weekday_table["share of synchronizations (%)"] = (
    weekday_table["synchronizations"] / len(events) * 100
)
display(weekday_table)

reduction_percent = (reference - summary["total_messages"]) / reference * 100
display(Markdown(
    f"Across {summary['bucket_count']:,} evaluated minutes, {quiet_minutes:,} were message-free and "
    f"{active_minutes:,} contained at least one transmission. The monitor sent {summary['total_messages']:,} "
    f"transmissions ({reduction_percent:.2f}% fewer than the {reference:,}-transmission centralized reference) "
    f"across {summary['resynchronization_count']:,} completed root synchronizations, including initialization. "
    "This is an event-driven transmission-count comparison: the centralized reference counts one sensor report "
    "per sensor per minute, whereas Phase 2 includes local triggers, requests, responses, and state updates. "
    "It does not establish protocol-equivalent state delivery or quantify bytes, latency, energy, or computation."
))

## 8. Existing interpretation and limitations

The matched run reports its sampled-bucket root-state correctness separately from its communication schedule. A zero mismatch count is bounded to the evaluated one-minute buckets and this fixed cohort; it is not a guarantee between samples or for individual sensors. The centralized comparison is a reference-count comparison only, not a causal attribution. Existing FMIC/SMIC comparisons remain preliminary because moment weighting and coordinator/cluster protocols are not controlled. The historical signed-margin result is deliberately excluded from current conclusions and denominators.

In [ ]:
display(pd.DataFrame({"Existing limitation": summary["limitations"]}))
display(Markdown(
    "**Read-only completion:** validated saved summary, matched trace, and synchronization-event evidence were displayed; "
    "no monitor, regression suite, experiment, or artifact export was run."
))